In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Data
df = pd.read_csv("ufc-master.csv")

# 2. Target Engineering
df['BlueWin'] = (df['Winner'] == 'Blue').astype(int)

# 3. Smart Feature Selection
# We exclude leakage columns but keep odds (as they are known pre-fight)
exclude_cols = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'Date', 'Location', 'Country', 'TitleBout', 'EmptyArena', 'Gender']
leak_keywords = ['Finish', 'TotalFightTime', 'Referee', 'Round', 'weightRank', 'FPRank', 'WinsBy'] # stricter leakage filtering

# Identify numeric columns
cols = df.columns.tolist()
feature_cols = [c for c in cols if c not in exclude_cols and not any(k in c for k in leak_keywords)]

# Separate numeric and categorical
numeric_cols = []
categorical_cols = ['WeightClass', 'BlueStance', 'RedStance', 'BetterRank'] 

for c in feature_cols:
    if c in categorical_cols:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_cols.append(c)

# 4. Advanced Preprocessing (The "Debutant" Logic)
X = df.copy()

# Handle Categorical: Fill NaN with 'Unknown' then Encode
for col in categorical_cols:
    X[col] = X[col].fillna('Unknown')
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Handle Numeric: Fill NaN with 0 (assuming missing stats = debutant/no data) 
# and Create "IsMissing" flag for key stats
# for col in ['BlueAvgSigStrLanded', 'RedAvgSigStrLanded', 'BlueAvgTDLanded', 'RedAvgTDLanded']:
#     X[f'{col}_missing'] = X[col].isna().astype(int)
#     numeric_cols.append(f'{col}_missing')

X[numeric_cols] = X[numeric_cols].fillna(0)

# Log transform highly skewed features (like total rounds fought) to normalize distribution
for col in ['BlueTotalRoundsFought', 'RedTotalRoundsFought', 'BlueTotalTitleBouts', 'RedTotalTitleBouts']:
    if col in numeric_cols:
        X[col] = np.log1p(X[col])

# Final Feature Set
X_final = X[numeric_cols + categorical_cols]
y_final = df['BlueWin']

# Standard Scaling for Neural Net
scaler = StandardScaler()
X_final[numeric_cols] = scaler.fit_transform(X_final[numeric_cols])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42, stratify=y_final)

print(f"Input Features: {X_train.shape}")
print(f"Categorical Features: {categorical_cols}")
print(f"Numeric Features: {numeric_cols}")

Input Features: (5232, 62)
Categorical Features: ['WeightClass', 'BlueStance', 'RedStance', 'BetterRank']
Numeric Features: ['RedOdds', 'BlueOdds', 'RedExpectedValue', 'BlueExpectedValue', 'BlueCurrentLoseStreak', 'BlueCurrentWinStreak', 'BlueDraws', 'BlueAvgSigStrLanded', 'BlueAvgSigStrPct', 'BlueAvgSubAtt', 'BlueAvgTDLanded', 'BlueAvgTDPct', 'BlueLongestWinStreak', 'BlueLosses', 'BlueTotalTitleBouts', 'BlueWins', 'BlueHeightCms', 'BlueReachCms', 'BlueWeightLbs', 'RedCurrentLoseStreak', 'RedCurrentWinStreak', 'RedDraws', 'RedAvgSigStrLanded', 'RedAvgSigStrPct', 'RedAvgSubAtt', 'RedAvgTDLanded', 'RedAvgTDPct', 'RedLongestWinStreak', 'RedLosses', 'RedTotalTitleBouts', 'RedWins', 'RedHeightCms', 'RedReachCms', 'RedWeightLbs', 'RedAge', 'BlueAge', 'LoseStreakDif', 'WinStreakDif', 'LongestWinStreakDif', 'WinDif', 'LossDif', 'TotalTitleBoutDif', 'KODif', 'SubDif', 'HeightDif', 'ReachDif', 'AgeDif', 'SigStrDif', 'AvgSubAttDif', 'AvgTDDif', 'BMatchWCRank', 'RMatchWCRank', 'RedDecOdds', 'BlueD

/var/folders/60/hcdjzdqj5tg0psxnbv_sjnwc0000gn/T/ipykernel_466/2367086096.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_final[numeric_cols] = scaler.fit_transform(X_final[numeric_cols])


In [99]:
ml_model_results = []
from xgboost import XGBClassifier

# Initialize XGBoost with optimized settings for tabular data
# derived from general best practices for sports data
xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(
    X_train, y_train, 
    eval_set=[(X_test, y_test)], 
    verbose=0
)

y_pred_xgb = xgb_model.predict(X_test)
ml_model_results.append({'model': 'XGBoost', 'accuracy': accuracy_score(y_test, y_pred_xgb)})

In [100]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
ml_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500, random_state=42)
}

In [101]:
for name, model in ml_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    # print(f"--- {name} Results ---")
    # print(f"Accuracy: {accuracy:.4f}")
    ml_model_results.append({'model': name, 'accuracy': accuracy})

display(pd.DataFrame(ml_model_results).set_index('model').sort_values(by='accuracy', ascending=False))

,accuracy
model,
GradientBoosting,0.661574
XGBoost,0.659282
LogisticRegression,0.656226
RandomForest,0.650115
MLP,0.580596


In [102]:
# --- 1. Dataset Class ---
class UFCDataset(Dataset):
    def __init__(self, X, y):
        self.X_num = torch.tensor(X[numeric_cols].values, dtype=torch.float32)
        self.X_cat = torch.tensor(X[categorical_cols].values, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

train_dataset = UFCDataset(X_train, y_train)
test_dataset = UFCDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# --- 2. The ResNet-Style Tabular Model ---
class ResidualBlock(nn.Module):
    def __init__(self, features, dropout_rate=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(features, features),
            nn.BatchNorm1d(features),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(features, features),
            nn.BatchNorm1d(features)
        )
        self.relu = nn.ReLU()
        
    def forward(self, x):
        residual = x
        out = self.block(x)
        out += residual # Skip Connection
        return self.relu(out)

class DeepUFCNet(nn.Module):
    def __init__(self, num_numeric, cat_dims, embedding_dims, hidden_units=[256, 128]):
        super().__init__()
        self.param_groups = [{'hidden_units': hidden_units}]
        
        # Embedding Layers for Categorical Data
        self.embeddings = nn.ModuleList([
            nn.Embedding(num, dim) for num, dim in zip(cat_dims, embedding_dims)
        ])
        total_emb_dim = sum(embedding_dims)
        
        # Input Layer
        input_dim = num_numeric + total_emb_dim
        self.input_bn = nn.BatchNorm1d(num_numeric) # Normalize numeric inputs immediately
        
        # Initial projection
        self.initial_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_units[0]),
            nn.BatchNorm1d(hidden_units[0]),
            nn.ReLU()
        )
        
        # Residual Blocks
        self.res_blocks = nn.ModuleList()
        for i in range(len(hidden_units)-1):
            self.res_blocks.append(
                nn.Sequential(
                    ResidualBlock(hidden_units[i]),
                    nn.Linear(hidden_units[i], hidden_units[i+1]),
                    nn.BatchNorm1d(hidden_units[i+1]),
                    nn.ReLU()
                )
            )
            
        # Output Head
        self.output = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_units[-1], 1),
            nn.Sigmoid()
        )
        
    def forward(self, x_num, x_cat):
        # Process Embeddings
        x_cat_list = []
        for i, emb in enumerate(self.embeddings):
            x_cat_list.append(emb(x_cat[:, i]))
        x_cat_combined = torch.cat(x_cat_list, 1)
        
        # Process Numeric
        x_num = self.input_bn(x_num)
        
        # Concatenate
        x = torch.cat([x_num, x_cat_combined], 1)
        
        # Pass through network
        x = self.initial_layer(x)
        for block in self.res_blocks:
            x = block(x)
            
        return self.output(x)

# --- 3. Setup Model Parameters ---
# Calculate embedding sizes (rule of thumb: min(50, num_categories/2))
cat_dims = [len(X_final[col].unique()) for col in categorical_cols]
emb_dims = [min(50, (x + 1) // 2) for x in cat_dims]

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# model = DeepUFCNet(
#     num_numeric=len(numeric_cols),
#     cat_dims=cat_dims,
#     embedding_dims=emb_dims,
#     hidden_units=[512, 256, 128] # Deeper and wider
# ).to(device)

# optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
results = []

# --- 4. Training Loop ---
def train_model(epochs, model, optimizer):
    # epochs = 30
    best_acc = 0
    hidden_units = model.param_groups[0]['hidden_units']
    lr = optimizer.param_groups[0]['lr']

    print(f"Training Neural Network on {device}...")

    for epoch in range(1,epochs+1):
        model.train()
        train_loss = 0
        
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            
            optimizer.zero_grad()
            y_pred = model(x_num, x_cat)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Evaluation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x_num, x_cat, y in test_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                preds = model(x_num, x_cat)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        
        val_preds = [1 if p > 0.5 else 0 for p in all_preds]
        val_acc = accuracy_score(all_labels, val_preds)
        
        scheduler.step(val_acc)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_at_epoch = epoch
            # Save best model logic here
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")

    results.append({'hidden_units': hidden_units, 'best accuracy': best_acc, 'best_at_epoch': best_at_epoch, 'lr': lr})
    print(f"Final Best Neural Net Accuracy: {best_acc:.4f} at epoch {best_at_epoch}")

In [103]:
model = DeepUFCNet(
    num_numeric=len(numeric_cols),
    cat_dims=cat_dims,
    embedding_dims=emb_dims,
    hidden_units=[512, 256, 128] # Deeper and wider
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

train_model(epochs=50,model=model,optimizer=optimizer)

Training Neural Network on mps...
Epoch 1 | Loss: 0.6461 | Val Acc: 0.6623
Epoch 5 | Loss: 0.5764 | Val Acc: 0.6409
Epoch 10 | Loss: 0.4849 | Val Acc: 0.6364
Epoch 15 | Loss: 0.3965 | Val Acc: 0.6073
Epoch 20 | Loss: 0.2942 | Val Acc: 0.5867
Epoch 25 | Loss: 0.2162 | Val Acc: 0.6012
Epoch 30 | Loss: 0.1697 | Val Acc: 0.5997
Epoch 35 | Loss: 0.1379 | Val Acc: 0.6066
Epoch 40 | Loss: 0.1250 | Val Acc: 0.5898
Epoch 45 | Loss: 0.1078 | Val Acc: 0.6005
Epoch 50 | Loss: 0.0883 | Val Acc: 0.6043
Final Best Neural Net Accuracy: 0.6631 at epoch 2


In [104]:
model = DeepUFCNet(
    num_numeric=len(numeric_cols),
    cat_dims=cat_dims,
    embedding_dims=emb_dims,
    hidden_units=[256, 128] # Deeper and wider
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

train_model(epochs=50,model=model,optimizer=optimizer)

Training Neural Network on mps...
Epoch 1 | Loss: 0.6417 | Val Acc: 0.6494
Epoch 5 | Loss: 0.5634 | Val Acc: 0.6241
Epoch 10 | Loss: 0.4906 | Val Acc: 0.6104
Epoch 15 | Loss: 0.3940 | Val Acc: 0.6073
Epoch 20 | Loss: 0.2998 | Val Acc: 0.5966
Epoch 25 | Loss: 0.2463 | Val Acc: 0.5905
Epoch 30 | Loss: 0.2031 | Val Acc: 0.6066
Epoch 35 | Loss: 0.1649 | Val Acc: 0.6050
Epoch 40 | Loss: 0.1668 | Val Acc: 0.5936
Epoch 45 | Loss: 0.1337 | Val Acc: 0.5959
Epoch 50 | Loss: 0.1086 | Val Acc: 0.5928
Final Best Neural Net Accuracy: 0.6555 at epoch 2


In [105]:
# Final Best Neural Net Accuracy: 0.6654 at epoch 4 | lr = 0.001
model = DeepUFCNet(
    num_numeric=len(numeric_cols),
    cat_dims=cat_dims,
    embedding_dims=emb_dims,
    hidden_units=[128, 64] # Deeper and wider
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

train_model(epochs=50,model=model,optimizer=optimizer)

Training Neural Network on mps...
Epoch 1 | Loss: 0.6374 | Val Acc: 0.6646
Epoch 5 | Loss: 0.5809 | Val Acc: 0.6532
Epoch 10 | Loss: 0.5297 | Val Acc: 0.6196
Epoch 15 | Loss: 0.4623 | Val Acc: 0.6188
Epoch 20 | Loss: 0.4149 | Val Acc: 0.6257
Epoch 25 | Loss: 0.3606 | Val Acc: 0.5966
Epoch 30 | Loss: 0.3091 | Val Acc: 0.6073
Epoch 35 | Loss: 0.2829 | Val Acc: 0.6005
Epoch 40 | Loss: 0.2580 | Val Acc: 0.6058
Epoch 45 | Loss: 0.2252 | Val Acc: 0.5943
Epoch 50 | Loss: 0.2012 | Val Acc: 0.5959
Final Best Neural Net Accuracy: 0.6646 at epoch 1


In [106]:
model = DeepUFCNet(
    num_numeric=len(numeric_cols),
    cat_dims=cat_dims,
    embedding_dims=emb_dims,
    hidden_units=[64,32] # Deeper and wider
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

train_model(epochs=50,model=model,optimizer=optimizer)

Training Neural Network on mps...
Epoch 1 | Loss: 0.6481 | Val Acc: 0.6494
Epoch 5 | Loss: 0.5984 | Val Acc: 0.6547
Epoch 10 | Loss: 0.5643 | Val Acc: 0.6471
Epoch 15 | Loss: 0.5291 | Val Acc: 0.6325
Epoch 20 | Loss: 0.5027 | Val Acc: 0.6280
Epoch 25 | Loss: 0.4675 | Val Acc: 0.6142
Epoch 30 | Loss: 0.4434 | Val Acc: 0.6142
Epoch 35 | Loss: 0.4167 | Val Acc: 0.6028
Epoch 40 | Loss: 0.3983 | Val Acc: 0.6005
Epoch 45 | Loss: 0.3776 | Val Acc: 0.6058
Epoch 50 | Loss: 0.3542 | Val Acc: 0.6112
Final Best Neural Net Accuracy: 0.6631 at epoch 8


In [107]:
# Final Best Neural Net Accuracy: 0.6677 at epoch 15 | lr = 0.001 
model = DeepUFCNet(
    num_numeric=len(numeric_cols),
    cat_dims=cat_dims,
    embedding_dims=emb_dims,
    hidden_units=[32, 16] # Deeper and wider
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

train_model(epochs=50,model=model,optimizer=optimizer)

Training Neural Network on mps...
Epoch 1 | Loss: 0.6713 | Val Acc: 0.6425
Epoch 5 | Loss: 0.6111 | Val Acc: 0.6524
Epoch 10 | Loss: 0.5896 | Val Acc: 0.6494
Epoch 15 | Loss: 0.5803 | Val Acc: 0.6516
Epoch 20 | Loss: 0.5606 | Val Acc: 0.6455
Epoch 25 | Loss: 0.5461 | Val Acc: 0.6455
Epoch 30 | Loss: 0.5270 | Val Acc: 0.6471
Epoch 35 | Loss: 0.5232 | Val Acc: 0.6379
Epoch 40 | Loss: 0.5058 | Val Acc: 0.6295
Epoch 45 | Loss: 0.5004 | Val Acc: 0.6287
Epoch 50 | Loss: 0.4894 | Val Acc: 0.6287
Final Best Neural Net Accuracy: 0.6600 at epoch 33


In [108]:
print("\nSummary:")
display(pd.DataFrame(results).set_index('hidden_units').sort_values('best accuracy', ascending=False))


Summary:


,best accuracy,best_at_epoch,lr
hidden_units,,,
"[128, 64]",0.664629,1,0.001
"[512, 256, 128]",0.663102,2,0.001
"[64, 32]",0.663102,8,0.001
"[32, 16]",0.660046,33,0.001
"[256, 128]",0.655462,2,0.001
